# Generate Gene Embeddings — ESM2

This notebook generates gene embeddings using the **ESM2** protein language model (`esm2_t33_650M_UR50D`, 650M params, layer 33 mean representation).

## Environment setup

Use env: `envs/esm2/` — install with:
```bash
cd /lustre/fswork/projects/rech/xeg/uat95fg/scPRINT
uv venv envs/esm2 --python 3.10
source envs/esm2/bin/activate
uv pip install fair-esm torch pandas biopython tqdm
uv pip install -e . --no-deps
```

Then launch this notebook with the `envs/esm2` kernel.

**Note:** The actual embedding generation is GPU-intensive. Submit as a SLURM job (see bottom of notebook) or run interactively on a GPU node.

In [ ]:
import os
import pandas as pd
from scdataloader import utils
from scprint.tokenizers.embedder import protein_embeddings_generator

os.makedirs('../data/main/gene_embs_esm2', exist_ok=True)

## Human genes (NCBITaxon:9606)

In [ ]:
hgenedf = utils.load_genes(organisms='NCBITaxon:9606')
print(f'Loaded {len(hgenedf)} human genes')
hgenedf.head()

In [ ]:
homo_emb, homo_naming_df = protein_embeddings_generator(
    hgenedf,
    organism='homo_sapiens',
    embedder='esm2',
    embedding_size=1152,
    cache=True,
    cuda=True,
)
print(f'Human embeddings shape: {homo_emb.shape}')
homo_emb.to_parquet('../data/main/gene_embs_esm2/homo_emb.parquet')
print('Saved homo_emb.parquet')

## Mouse genes (NCBITaxon:10090)

In [ ]:
mgenedf = utils.load_genes(organisms='NCBITaxon:10090')
print(f'Loaded {len(mgenedf)} mouse genes')
mgenedf.head()

In [ ]:
mus_emb, mus_naming_df = protein_embeddings_generator(
    mgenedf,
    organism='mus_musculus',
    embedder='esm2',
    embedding_size=1152,
    cache=True,
    cuda=True,
)
print(f'Mouse embeddings shape: {mus_emb.shape}')
mus_emb.to_parquet('../data/main/gene_embs_esm2/mus_emb.parquet')
print('Saved mus_emb.parquet')

## Concatenate and save final embeddings

In [ ]:
# Load from disk in case cells were run in separate jobs
homo_emb = pd.read_parquet('../data/main/gene_embs_esm2/homo_emb.parquet')
mus_emb  = pd.read_parquet('../data/main/gene_embs_esm2/mus_emb.parquet')

gene_embeddings = pd.concat([homo_emb, mus_emb])
# Drop duplicate gene IDs (keep first)
gene_embeddings = gene_embeddings[~gene_embeddings.index.duplicated(keep='first')]
print(f'Combined embeddings shape: {gene_embeddings.shape}')

gene_embeddings.to_parquet('../data/main/gene_embs_esm2/gene_embeddings.parquet')
print('Saved gene_embeddings.parquet')

## UMAP visualisation

In [ ]:
import umap
import matplotlib.pyplot as plt
import numpy as np

gene_embeddings = pd.read_parquet('../data/main/gene_embs_esm2/gene_embeddings.parquet')

reducer = umap.UMAP(n_components=2, random_state=42)
embedding_2d = reducer.fit_transform(gene_embeddings.values)

umap_df = pd.DataFrame(
    embedding_2d, columns=['UMAP1', 'UMAP2'], index=gene_embeddings.index
)

is_hum = umap_df.index.str.startswith('ENSG')

plt.figure(figsize=(10, 8))
plt.scatter(umap_df.loc[~is_hum, 'UMAP1'], umap_df.loc[~is_hum, 'UMAP2'],
            alpha=0.5, s=1, label='Mouse', color='blue')
plt.scatter(umap_df.loc[is_hum, 'UMAP1'], umap_df.loc[is_hum, 'UMAP2'],
            alpha=0.5, s=1, label='Human', color='red')
plt.title('UMAP projection of ESM2 gene embeddings')
plt.xlabel('UMAP1')
plt.ylabel('UMAP2')
plt.legend()
plt.tight_layout()
plt.savefig('../data/main/gene_embs_esm2/umap.png', dpi=150)
plt.show()

## SLURM submission (optional)

To run the embedding generation as a SLURM job on Jean Zay:

```bash
#!/bin/bash
#SBATCH --job-name=esm2_emb
#SBATCH --ntasks=1
#SBATCH --cpus-per-task=8
#SBATCH --gres=gpu:1
#SBATCH --time=04:00:00
#SBATCH --account=xeg@v100

cd /lustre/fswork/projects/rech/xeg/uat95fg/scPRINT
source envs/esm2/bin/activate
jupyter nbconvert --to notebook --execute notebooks/generate_gene_embeddings_esm2.ipynb \
    --output notebooks/generate_gene_embeddings_esm2_executed.ipynb
```